# Part A: Dataset Structure
## 1. Data Loading & Column Inventory
Loading the datasets to inspect their shape, columns, and data types.

In [1]:
import pandas as pd
import os

# Load datasets
caract = pd.read_csv('caract-2024.csv', sep=';')
lieux = pd.read_csv('lieux-2024.csv', sep=';')
usagers = pd.read_csv('usagers-2024.csv', sep=';')
vehicules = pd.read_csv('vehicules-2024.csv', sep=';')

datasets = {'caract': caract, 'lieux': lieux, 'usagers': usagers, 'vehicules': vehicules}


In [2]:
for name, df in datasets.items():
    print(f"--- {name.upper()} ---")
    print(f"Shape: {df.shape}")
    print("\nData Types:")
    print(df.dtypes)
    print("\n")


--- CARACT ---
Shape: (54402, 15)

Data Types:
Num_Acc     int64
jour        int64
mois        int64
an          int64
hrmn       object
lum         int64
dep        object
com        object
agg         int64
int         int64
atm         int64
col         int64
adr        object
lat        object
long       object
dtype: object


--- LIEUX ---
Shape: (70248, 18)

Data Types:
Num_Acc     int64
catr        int64
voie       object
v1          int64
v2         object
circ        int64
nbv        object
vosp        int64
prof        int64
pr         object
pr1        object
plan        int64
lartpc     object
larrout    object
surf        int64
infra       int64
situ        int64
vma         int64
dtype: object


--- USAGERS ---
Shape: (125187, 16)

Data Types:
Num_Acc          int64
id_usager       object
id_vehicule     object
num_veh         object
place            int64
catu             int64
grav             int64
sexe             int64
an_nais        float64
trajet           int64
se

## 2. Semantic Meaning
### Characteristics (`caract-2024.csv`)
- `Num_Acc`: Unique accident identifier
- `jour`, `mois`, `an`: Date of the accident
- `hrmn`: Time of accident
- `lum`: Lighting conditions
- `dep`, `com`: Department and Municipality
- `agg`: Location type (built-up area vs out)
- `int`: Intersection type
- `atm`: Atmospheric conditions
- `col`: Collision type
- `adr`, `lat`, `long`: Address and coordinates

### Locations (`lieux-2024.csv`)
- `Num_Acc`: Unique accident identifier
- `catr`: Road category
- `voie`, `v1`, `v2`: Route/Road identifiers
- `circ`: Traffic regime
- `nbv`: Number of lanes
- `vosp`: Reserved lanes
- `prof`, `plan`: Road profile and plan
- `pr`, `pr1`: Home reference point
- `lartpc`, `larrout`: Road width
- `surf`: Surface condition
- `infra`, `situ`: Infrastructure and situation
- `vma`: Max speed

### Users (`usagers-2024.csv`)
- `Num_Acc`, `id_usager`, `id_vehicule`, `num_veh`: IDs
- `place`: Seat position
- `catu`: User category (driver, passenger, pedestrian)
- `grav`: Severity of injury
- `sexe`, `an_nais`: Gender and birth year
- `trajet`: Reason for travel
- `secu1`, `secu2`, `secu3`: Safety equipment
- `locp`, `actp`, `etatp`: Pedestrian info

### Vehicles (`vehicules-2024.csv`)
- `Num_Acc`, `id_vehicule`, `num_veh`: IDs
- `senc`: Direction of travel
- `catv`: Vehicle category
- `obs`, `obsm`: Obstacles hit
- `choc`: Initial point of impact
- `manv`: Main maneuver
- `motor`: Motorization type
- `occutc`: Number of occupants (public transport)

# Part B: Missing Values and Completeness
In this section, we compute the percentage of missing values (including standard NaNs and `-1` values which represent 'Unknown' in this dataset), identify critical missingness, and suggest remediation strategies.

In [3]:
import numpy as np

# Function to summarize missing values including NaNs and -1
def summarize_missingness(df):
    # Standard missing (NaNs)
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    
    # Unknowns (-1)
    unknowns_pct = pd.Series(0.0, index=df.columns)
    for col in df.columns:
        if df[col].dtype in ['int64', 'float64', 'int32', 'float32']:
            unknowns_pct[col] = (df[col] == -1).sum() / len(df) * 100
            
    missing_df = pd.DataFrame({
        'NaN_Pct': missing_pct,
        'Unknown_Pct(-1)': unknowns_pct,
        'Total_Missing_Pct': missing_pct + unknowns_pct
    })
    
    # Filter to only show columns with > 0% missingness
    return missing_df[missing_df['Total_Missing_Pct'] > 0].sort_values(by='Total_Missing_Pct', ascending=False)

for name, df in datasets.items():
    print(f"--- {name.upper()} Missingness ---")
    display(summarize_missingness(df))
    print("\n")


--- CARACT Missingness ---


,NaN_Pct,Unknown_Pct(-1),Total_Missing_Pct
adr,4.246167,0.000000,4.246167
col,0.000000,0.011029,0.011029




--- LIEUX Missingness ---


,NaN_Pct,Unknown_Pct(-1),Total_Missing_Pct
lartpc,99.953024,0.000000,99.953024
v2,91.578408,0.000000,91.578408
v1,0.000000,23.163649,23.163649
voie,18.977053,0.000000,18.977053
circ,0.000000,6.198041,6.198041
vosp,0.000000,5.454960,5.454960
vma,0.000000,5.167407,5.167407
infra,0.000000,1.155905,1.155905
prof,0.000000,0.071176,0.071176
plan,0.000000,0.056941,0.056941




--- USAGERS Missingness ---


,NaN_Pct,Unknown_Pct(-1),Total_Missing_Pct
etatp,0.000000,91.808255,91.808255
secu3,0.000000,90.371205,90.371205
locp,0.000000,49.342983,49.342983
secu2,0.000000,42.986093,42.986093
trajet,0.000000,2.097662,2.097662
an_nais,2.060118,0.000000,2.060118
sexe,0.000000,1.913138,1.913138
secu1,0.000000,1.679887,1.679887
place,0.000000,0.002396,0.002396




--- VEHICULES Missingness ---


,NaN_Pct,Unknown_Pct(-1),Total_Missing_Pct
occutc,98.976025,0.000000,98.976025
motor,0.000000,0.207169,0.207169
senc,0.000000,0.073372,0.073372
choc,0.000000,0.047476,0.047476
obsm,0.000000,0.032370,0.032370
obs,0.000000,0.029133,0.029133
manv,0.000000,0.029133,0.029133
catv,0.000000,0.001079,0.001079


## Critical Missingness Analysis

Based on the output above, here are the most problematic missing values:

1. **High Missingness (> 90%)**: Columns like `lartpc` (width of central divider), `v2` (alphanumeric route index), `secu3` (tertiary safety equipment), `etatp` (pedestrian status), and `occutc` (public transport occupants). 
   - *Why problematic:* They are virtually empty for the vast majority of the dataset.
   - *Context:* These are often conditionally applicable (e.g., `occutc` only applies if the vehicle is public transport, `etatp` only applies to pedestrians).

2. **Moderate Missingness (10% - 50%)**: Columns like `voie` (18.9%), `v1` (23.1% unknown), `secu2` (42.9%), `locp` (49.3%).
   - *Why problematic:* `voie` and `v1` are crucial for identifying the exact road where the accident happened. `locp` is critical for pedestrian accident analysis.

3. **Low but Critical Missingness (< 10%)**: `vma` (max speed, 5.1% unknown), `adr` (address, 4.2%), `an_nais` (birth year, 2.0%).
   - *Why problematic:* `vma` and `an_nais` are highly predictive features for accident severity. Losing age or speed limit data can bias predictive models.

## Remediation Strategies

1. **Standardize Missing Values**: First, replace all `-1` values with `np.nan` so that pandas can universally handle them as missing data across all operations.
2. **Conditional Columns (> 90% missing)**: Do not drop these columns if analyzing specific sub-groups (e.g. pedestrians, public transport). Instead, fill missing values with a designated category like `"Not Applicable"` or `0` depending on the context.
3. **Categorical Imputation**: For features like `vma` (max speed) or `lum` (lighting), impute using the **mode** (most frequent value) based on the road category (`catr`) or intersection type (`int`).
4. **Numerical Imputation**: For `an_nais` (birth year), impute missing values using the **median** birth year, possibly grouped by user category (`catu`).
5. **Row Deletion**: For critical identifiers like `adr` (address) or coordinates, if location analysis is the primary goal and imputation is impossible, dropping the rows (only ~4%) might be the safest approach to prevent plotting accidents in the ocean (0,0).


# Part C: Consistency and Validity Checks
In this section, we analyze the dataset for out-of-range values, invalid categories, and duplicate records.

In [4]:
import pandas as pd

print("--- 1. DUPLICATES ---")
for name, df in datasets.items():
    duplicates = df.duplicated().sum()
    print(f"{name.capitalize()}: {duplicates} duplicates")

print("\n--- 2. VALUE RANGES (Anomalies) ---")
if 'an_nais' in usagers.columns:
    invalid_birth_years = usagers[(usagers['an_nais'] < 1900) | (usagers['an_nais'] > 2024)]
    print(f"Usagers (an_nais): {len(invalid_birth_years)} rows with invalid birth year (<1900 or >2024)")

caract['lat_float'] = pd.to_numeric(caract['lat'].str.replace(',', '.'), errors='coerce')
caract['long_float'] = pd.to_numeric(caract['long'].str.replace(',', '.'), errors='coerce')

invalid_lat = caract[(caract['lat_float'] < -90) | (caract['lat_float'] > 90)]
invalid_long = caract[(caract['long_float'] < -180) | (caract['long_float'] > 180)]
zero_coords = caract[(caract['lat_float'] == 0) & (caract['long_float'] == 0)]

print(f"Caract (Coordinates): {len(invalid_lat)} invalid latitudes (< -90 or > 90)")
print(f"Caract (Coordinates): {len(invalid_long)} invalid longitudes (< -180 or > 180)")
print(f"Caract (Coordinates): {len(zero_coords)} default (0,0) coordinates")

print("\n--- 3. CATEGORICAL ANOMALIES ---")
invalid_sex = usagers[~usagers['sexe'].isin([1, 2, -1])]['sexe'].value_counts()
if len(invalid_sex) > 0:
    print(f"Usagers (sexe): Invalid categories found:\n{invalid_sex}")
else:
    print("Usagers (sexe): No invalid categories (only 1, 2, -1 found).")

invalid_lum = caract[~caract['lum'].isin([1, 2, 3, 4, 5, -1])]['lum'].value_counts()
if len(invalid_lum) > 0:
    print(f"Caract (lum): Invalid categories found:\n{invalid_lum}")
else:
    print("Caract (lum): No invalid categories (only 1-5, -1 found).")


--- 1. DUPLICATES ---
Caract: 0 duplicates
Lieux: 2 duplicates
Usagers: 0 duplicates
Vehicules: 0 duplicates

--- 2. VALUE RANGES (Anomalies) ---
Usagers (an_nais): 0 rows with invalid birth year (<1900 or >2024)
Caract (Coordinates): 0 invalid latitudes (< -90 or > 90)
Caract (Coordinates): 0 invalid longitudes (< -180 or > 180)
Caract (Coordinates): 0 default (0,0) coordinates

--- 3. CATEGORICAL ANOMALIES ---
Usagers (sexe): No invalid categories (only 1, 2, -1 found).
Caract (lum): No invalid categories (only 1-5, -1 found).


## Consistency Findings

The dataset exhibits high consistency and validity:

1. **Duplicates**: 
   - Only **2 duplicates** were found in the `lieux` (Locations) dataset.
   - The `caract`, `usagers`, and `vehicules` tables are 100% duplicate-free.
   - **Remediation**: Use `df.drop_duplicates()` on the `lieux` table.

2. **Value Ranges**:
   - **Ages**: There are no negative ages, and no birth years before 1900 or after 2024.
   - **Coordinates**: The coordinates are mathematically valid (Latitudes between -90 and +90, Longitudes between -180 and +180). There are also no default `(0,0)` coordinate anomalies.

3. **Categorical Anomalies**:
   - Core categorical variables like gender (`sexe`) and lighting (`lum`) conform perfectly to their expected schema dictionaries (`1, 2, -1` for gender; `1-5, -1` for lighting). There are no unexpected outliers.
